# Drone RL — independent PPO, DQN, and SAC results

To view a saved result, run **Setup**, **Load the RNN**, **Choose a mission**,
and **Plot helpers**, then
run only the desired enabled algorithm's **Results** cell. No training cell or other
algorithm's results cell is required. Checkpoint paths are editable in Setup.
Set `SKIP_DQN` / `SKIP_SAC` in Setup; both default to `True` and skip training
and results, so Run All runs PPO only. Choose the mission before creating environments.

Each algorithm has an optional training cell, its own flight plot,
and its own learning curve. Physical state stays `[z, vz, az]`; the policy observes
`[z-z_ref, vz, az]`. Training, evaluation, plots, and exports use the selected mission.


## Setup


In [ ]:
# Run once if needed, then restart the kernel:
# %pip install gymnasium stable-baselines3 numpy matplotlib scipy

import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import gymnasium as gym
from stable_baselines3 import PPO, DQN, SAC
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.results_plotter import load_results

PROJECT_ROOT = next(
    (candidate for directory in (Path.cwd(), *Path.cwd().parents)
     for candidate in (directory, directory / "EI-DATA-DRIVEN-CONTROL")
     if (candidate / "src/model/rnn_loader.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this project or src/policy/notebooks/ to find src/model/rnn_loader.py")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Choose "z_angles" for rotation, or "z" for the previous altitude model.
RNN_VARIANT = "z_angles"
if RNN_VARIANT == "z_angles":
    from policy.lib.droneEnvRotation import DroneEnv, PARAMS, Z_LAND, default_mission, run_episode, evaluate
elif RNN_VARIANT == "z":
    from policy.lib.droneEnv import DroneEnv, PARAMS, Z_LAND, default_mission, run_episode, evaluate
else:
    raise ValueError("RNN_VARIANT must be 'z_angles' or 'z'")
from model.rnn_loader import load_rnn_model
from policy.training import load_ppo_config, train_ppo

from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

PPO_CONFIG_PATH = PROJECT_ROOT / "src/policy/config/ppo.json"
PPO_CONFIG = load_ppo_config(PPO_CONFIG_PATH)
SEED = PPO_CONFIG["seed"]
# Skip both training and results for these algorithms during Run All.
SKIP_DQN = True
SKIP_SAC = True
torch.set_num_threads(1)
# Saved results: edit one path to inspect a different checkpoint.
MODEL_PATH = PROJECT_ROOT / "src" / "policy" / "checkpoints" / ("ppo_drone_z_angles.zip" if RNN_VARIANT == "z_angles" else "ppo_drone_rnn.zip")
PPO_MODEL_PATH = MODEL_PATH
DQN_MODEL_PATH = PROJECT_ROOT / 'src/policy/logs/tuning_dqn_multistep_20261103_163054/best_model.zip'
SAC_MODEL_PATH = PROJECT_ROOT / 'src/policy/logs/tuning_sac_smooth_20261103_163155/best_model.zip'

# New training runs get separate folders; plotting does not create or require them.
RUN_DIR = PROJECT_ROOT / "results" / "policy" / f"rl_{RNN_VARIANT}_tuned_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_DIR = PROJECT_ROOT / "results" / "export_logs" / RNN_VARIANT
PPO_DIR = RUN_DIR / "ppo"
PPO_STEPS = PPO_CONFIG["total_timesteps"]
PPO_PARAMS = dict(PPO_CONFIG["ppo"])
# Base exploration remains broader; yaw differential is highly sensitive.
PPO_INITIAL_LOG_STD = (-1.5, -3.0) if RNN_VARIANT == "z_angles" else PPO_CONFIG["initial_log_std"]
DQN_DIR = RUN_DIR / "dqn"
DQN_STEPS = 500000
DQN_PARAMS = {'learning_rate': 0.0001,
 'buffer_size': 100000,
 'learning_starts': 1000,
 'batch_size': 128,
 'train_freq': 5,
 'gradient_steps': -1,
 'n_steps': 3,
 'target_update_interval': 1000,
 'tau': 0.1,
 'exploration_fraction': 1.0,
 'policy_kwargs': {'net_arch': [64, 64]}}
SAC_DIR = RUN_DIR / "sac"
SAC_STEPS = 500000
SAC_PARAMS = {'learning_rate': 0.0003,
 'buffer_size': 100000,
 'learning_starts': 1000,
 'batch_size': 128,
 'train_freq': 1,
 'gradient_steps': -1,
 'ent_coef': 'auto_0.05',
 'use_sde': True,
 'sde_sample_freq': 4,
 'use_sde_at_warmup': True,
 'policy_kwargs': {'net_arch': [64, 64], 'log_std_init': -2}}
SAC_ACTION_LIMIT = 0.8


## Load the trained RNN and head

`RNN_VARIANT="z_angles"` selects `src/model/checkpoints/rnn_model_z_and_angles` and the
rotation environment. The checkpoint has a 4-input, 32-hidden-unit, one-layer
RNN and a 32 → 64 → 4 head with Tanh between its linear layers.

Set `RNN_VARIANT="z"` to select the previous altitude environment and
`src/model/checkpoints/rnn_z`, whose head is a single linear layer.
The original model and training files are retained.

The loader reads dimensions from the weights and loads them strictly, without
running either training script. Extracted checkpoint files are repacked in
memory. Each environment keeps its own recurrent hidden state.


In [ ]:
model_path = PROJECT_ROOT / "src" / "model" / "checkpoints" / (
    "rnn_model_z_and_angles" if RNN_VARIANT == "z_angles" else "rnn_z"
)
acceleration_rnn = load_rnn_model(model_path, head_activation="tanh")

print("Model:", model_path)
print("Load: all checkpoint weights matched")
print("RNN:", acceleration_rnn.rnn)
print("Head:", acceleration_rnn.head)


## Choose a mission

Edit `ref_mission(t)` below. All environments use it for training, evaluation,
and export. Rerun this cell and recreate environments after editing it.

Rotation missions return `(z_ref, phi_ref, psi_ref)`. Theta tracking is commented
out; physical pitch still evolves and remains visible in the plots. The rotation
policy now has 11 observations, so previous 12-input checkpoints need retraining.

The policy receives only the current reference through its tracking errors;
the mission function and future targets are not policy inputs. Full reference
curves are used for plots and reference generation during MATLAB replay.


In [ ]:
def ref_mission(t):
    """Return (z_ref, phi_ref, psi_ref); theta tracking is disabled."""
    if RNN_VARIANT == "z":
        return 0.0 if t < 2.0 else (2.0 if t < 6.0 else Z_LAND)
    if t < 2.0:
        # return (0.0, 0.0, 0.0, 0.0)                 # Step 1: On the platform (z = 0)
        return (0.0, 0.0, 0.0)                 # Step 1: On the platform (z = 0)
    elif t < 5.0:
        # return (1.0, 0.0, 0.0, 0.0)                 # Step 2: Climb and hold altitude (z = 1.0)
        return (6.0, 0.0, 0.0)                 # Step 2: Climb and hold altitude (z = 1.0)
    elif t < 8.0:
        # return (1.0, 0.0, 0.0, float(2.0 * np.pi * (t - 5.0) / 3.0))  # Step 3: Hold altitude and complete 360° yaw turn
        return (6.0, 0.0, float(2.0 * np.pi * (t - 5.0) / 3.0))  # Step 3: Hold altitude and complete 360° yaw turn
    else:
        # return (Z_LAND, 0.0, 0.0, float(2.0 * np.pi))
        return (Z_LAND, 0.0, float(2.0 * np.pi))



## Plot helpers

Shared action wrappers and plotting functions; this cell does not train or load a policy.


In [ ]:
DQN_COMMANDS = np.linspace(-3.0, 3.0, 61, dtype=np.float32)

class DiscreteDroneActions(gym.ActionWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.Discrete(len(DQN_COMMANDS))

    def action(self, index):
        return np.array([DQN_COMMANDS[int(index)]], dtype=np.float32)

class LimitedCommands(gym.Wrapper):
    def __init__(self, env, limit):
        super().__init__(env)
        self.action_space = gym.spaces.Box(-limit, limit, env.action_space.shape, dtype=np.float32)

def show_result(name, checkpoint, mission_to_test = None):
    if callable(checkpoint):
        learned_policy = checkpoint
        model = None
        checkpoint = None
    else:
        checkpoint = Path(checkpoint)
        load_kwargs = {"replay_buffer_class": None} if name == "DQN" else {}
        model = {"PPO": PPO, "DQN": DQN, "SAC": SAC}[name].load(checkpoint, device="cpu", **load_kwargs)
        commands = np.load(checkpoint.parent / "commands.npy") if name == "DQN" else None

        def learned_policy(obs):
            action = model.predict(obs, deterministic=True)[0]
            return np.array([commands[int(action)]], dtype=np.float32) if commands is not None else action

    if mission_to_test is not None:
        eval_env = DroneEnv(rnn_model=acceleration_rnn, mission=mission_to_test)
    else:
        eval_env = DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission)
    if model is not None:
        print(f"{name}: {checkpoint} ({model.num_timesteps:,} training steps)")
    else:
        print(f"{name}: heuristic controller")
    states, actions, total, outcome = run_episode(eval_env, learned_policy, seed=100)
    base = eval_env.unwrapped
    mission = base.mission
    t = np.arange(len(states)) * base.p.dt
    u = np.asarray(actions).reshape(len(actions), -1)
    targets = np.asarray([mission(float(ti)) for ti in t])
    z_target = targets if targets.ndim == 1 else targets[:, 0]
    has_psi = states.shape[1] > 11
    mae = np.mean(np.abs(states[1:, 0] - z_target[1:]))
    rms_du = np.sqrt(np.mean(np.diff(np.vstack([np.zeros((1, u.shape[1])), u]), axis=0) ** 2))
    status = ", ".join(outcome) if isinstance(outcome, tuple) else outcome
    print(f"{name}: {status} at {t[-1]:.2f} s | return {total:.1f} | "
          f"MAE {mae:.3f} m | peak altitude {states[:, 0].max():.3f} m | "
          f"final vz {states[-1, 1]:.3f} m/s | RMS delta-u {rms_du:.3f}")
    if has_psi:
        print(f"Final psi: {states[-1, 11]:.3f} rad ({np.degrees(states[-1, 11]):.1f} deg)")
        print(f"Final pitch/roll: {np.degrees(states[-1, 9]):.1f} / "
              f"{np.degrees(states[-1, 10]):.1f} deg | "
              f"angular speed {np.linalg.norm(states[-1, [12, 14, 16]]):.3f} rad/s")
    horizon = base.max_steps * base.p.dt
    duration = t[-1]
    end = min(horizon, duration + 0.25) if duration < 2.0 else horizon
    fig, axes = plt.subplots(6 if has_psi else 3, 1,
                             figsize=(9, 12 if has_psi else 7), sharex=True)
    mask = t <= end
    line, = axes[0].plot(t[mask], states[mask, 0], label=f"{name}: {status} at {t[-1]:.2f} s")
    color = line.get_color()
    axes[1].plot(t[mask], states[mask, 1], color=color)
    final_values = [states[-1, 0], states[-1, 1]]
    ylabels = ["Altitude z (m)", "Velocity vz (m/s)"]
    if has_psi:
        # Physical angles from the environment; yaw keeps complete turns visible.
        angle_specs = [(11, "psi", "Yaw psi (rad)"),
                       (9, "theta", "Pitch theta (rad; uncontrolled)"),
                       (10, "phi", "Roll phi (rad)")]
        for ax, (index, angle, ylabel) in zip(axes[2:5], angle_specs):
            ax.plot(t[mask], states[mask, index], color=color, label=f"Environment {angle}")
            final_values.append(states[-1, index])
            ylabels.append(ylabel)
    padded_u = np.vstack([u, u[-1:]])
    command_labels = ["u_base", "u_diff"] if u.shape[1] == 2 else [f"u{i + 1}" for i in range(u.shape[1])]
    for channel, label in enumerate(command_labels):
        command_line, = axes[-1].step(t[mask], padded_u[mask, channel],
                                      where="post", label=label)
        if t[-1] <= end:
            axes[-1].plot(t[-1], u[-1, channel], "x", color=command_line.get_color())
    if u.shape[1] > 1:
        axes[-1].legend(fontsize=8)
    if t[-1] <= end:
        for ax, value in zip(axes[:-1], final_values):
            ax.plot(t[-1], value, "x", color=color)
        for ax in axes:
            ax.axvline(t[-1], color=color, linestyle=":", alpha=0.4)
    reference_t = np.arange(0, end + base.p.dt / 2, base.p.dt)
    reference = np.asarray([mission(float(ti)) for ti in reference_t])
    z_reference = reference if reference.ndim == 1 else reference[:, 0]
    axes[0].step(reference_t, z_reference, where="post",
                 color="black", linestyle="--", label="Mission target")
    if has_psi:
        # Rotation missions use (z, phi, psi); pitch is plotted without a target.
        # Legacy (z, psi) and (z, theta, phi, psi) missions are also accepted.
        # theta_reference = np.zeros_like(reference_t)
        phi_reference = np.zeros_like(reference_t)
        psi_reference = np.zeros_like(reference_t)
        if reference.ndim == 2:
            psi_reference = reference[:, -1]
            if reference.shape[1] == 3:
                phi_reference = reference[:, 1]
            elif reference.shape[1] == 4:
                # theta_reference = reference[:, 1]
                phi_reference = reference[:, 2]
        for ax, target, angle in zip([axes[2], axes[4]],
                                     [psi_reference, phi_reference],
                                     ["psi", "phi"]):
            ax.plot(reference_t, target, color="black", linestyle="--",
                    label=f"Mission {angle} target")
            ax.legend(fontsize=8)
    ylabels.append("Command u")
    for ax, ylabel in zip(axes, ylabels):
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    axes[0].set_title(f"{name} — saved policy" if model is not None else name)
    axes[0].legend(fontsize=8)
    axes[-1].set(xlabel="Time (s)", xlim=(0, end))
    fig.tight_layout()
    plt.show()
    eval_env.close()

    if checkpoint is not None and list(checkpoint.parent.glob("*.monitor.csv")):
        history = load_results(str(checkpoint.parent))
        returns = history["r"].to_numpy()
        steps = history["l"].cumsum().to_numpy()
        window = min(20, len(returns))
        fig, ax = plt.subplots(figsize=(9, 3))
        if window:
            ax.plot(steps[window - 1:], np.convolve(returns, np.ones(window) / window, "valid"),
                    label=f"Training: mean over {window} episodes", alpha=0.7)
        evaluation_file = checkpoint.parent / "evaluations.npz"
        if evaluation_file.is_file():
            with np.load(evaluation_file) as evaluations:
                ax.plot(evaluations["timesteps"], evaluations["results"].mean(axis=1),
                        marker=".", label="Deterministic evaluation")
        ax.axvline(model.num_timesteps, color="black", linestyle=":", label="Loaded checkpoint")
        ax.set(title=f"{name} learning curve", xlabel="Environment steps", ylabel="Episode return")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()

## Collective-only baseline: 2 m climb and descent

Run Setup, Load the RNN (`RNN_VARIANT="z_angles"`), and Plot helpers, then this
cell. The mission holds 0 m until 2 s, requests 2 m until 6 s, then requests the
landing altitude. Roll and yaw references are zero; theta tracking is disabled.

The PID reads only the current altitude error and vertical velocity. Its gains
are calibrated for the loaded rotation RNN in the environment's motor-command
units. An integral clamp and command slew limit keep the input smooth.
Every action is explicitly `[u_base, 0.0]`. The angle plots and touchdown checks
show whether collective input alone keeps the drone level.


In [ ]:
assert acceleration_rnn.rnn.input_size == 4, "Load the z_angles RNN before this cell."

def collective_mission(t):
    z_ref = 0.0 if t < 2.0 else (2.0 if t < 6.0 else Z_LAND)
    # return (z_ref, 0.0, 0.0, 0.0)
    return (z_ref, 0.0, 0.0)

def make_collective_heuristic():
    integral = 0.0
    previous = 0.0

    def policy(obs):
        nonlocal integral, previous
        error, vz = obs[:2]
        integral = float(np.clip(integral + float(error) * PARAMS.dt, -3.0, 3.0))
        u_base = -0.08 - 0.28 * error - 0.20 * vz - 0.01 * integral
        u_base = float(np.clip(u_base, previous - 0.08, previous + 0.08))
        u_base = float(np.clip(u_base, -1.0, 1.0))
        previous = u_base
        return np.array([u_base, 0.0], dtype=np.float32)

    return policy

show_result("Collective-only 2 m heuristic", make_collective_heuristic(),
            mission_to_test=collective_mission)


## Optional environment check


In [ ]:
env = DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission)
check_env(env)
obs, info = env.reset(seed=SEED)
print("Observation:", obs, env.observation_space)
print("Action:", env.action_space)
print("check_env: OK")
env.close()


## PPO — optional training

Settings come from `src/policy/config/ppo.json`. This cell uses the same
training function as `main_z.py` and saves TensorBoard logs and the resolved
config under `PPO_DIR`.


In [ ]:
start = time.perf_counter()
agent, PPO_MODEL_PATH = train_ppo(
    acceleration_rnn,
    ref_mission,
    PPO_DIR,
    total_timesteps=PPO_STEPS,
    seed=SEED,
    env_class=DroneEnv,
    initial_log_std=PPO_INITIAL_LOG_STD,
    config_path=PPO_CONFIG_PATH,
    **PPO_PARAMS,
)
print(f"PPO: {agent.num_timesteps:,}-step checkpoint; {time.perf_counter() - start:.1f} s")
print("Checkpoint:", PPO_MODEL_PATH)
print("TensorBoard logs:", PPO_DIR / "tensorboard")


## PPO — Results

Run this cell directly after the shared setup cells to load the saved PPO.


In [ ]:
show_result("PPO", PPO_MODEL_PATH)


In [ ]:
agent.save(MODEL_PATH)
print("Saved:", MODEL_PATH.with_suffix(".zip"))

# The existing MATLAB exporter handles the previous single-command model.
EXPORT_MATLAB = RNN_VARIANT == "z"
if EXPORT_MATLAB:
    from policy.export_policy import export_for_matlab
    export_env = DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission)
    export_for_matlab(agent, export_env, output_dir=LOG_DIR)
    export_env.close()

## Changing mission

In [ ]:
def test_mission(t):
    """Return (z_ref, phi_ref, psi_ref); theta tracking is disabled."""
    if RNN_VARIANT == "z":
        return 0.0 if t < 2.0 else (2.0 if t < 5.0 else Z_LAND)
    if t < 2.0:
        # return (0.0, 0.0, 0.0, 0.0)                 # Step 1: On the platform (z = 0)
        return (0.0, 0.0, 0.0)                 # Step 1: On the platform (z = 0)
    elif t < 5.0:
        # return (2.0, 0.0, 0.0, 0.0)                 # Step 2: Climb and hold altitude (z = 1.0)
        return (2.0, 0.0, 0.0)                 # Step 2: Climb and hold altitude (z = 1.0)
    elif t < 8.0:
        # return (2.0, 0.0, 0.0, float(2.0 * np.pi * (t - 5.0) / 3.0))  # Step 3: Hold altitude and complete 360° yaw turn
        return (2.0, 0.0, float(2.0 * np.pi * (t - 5.0) / 3.0))  # Step 3: Hold altitude and complete 360° yaw turn
    else:
        # return (Z_LAND, 0.0, 0.0, float(2.0 * np.pi))  
        return (Z_LAND, 0.0, float(2.0 * np.pi))  


In [ ]:
show_result("PPO", PPO_MODEL_PATH, mission_to_test=test_mission)

## DQN — optional training

Skipped when `SKIP_DQN=True` in Setup.

Use 61 commands evenly spaced across `[-3.0, 3.0]` (0.1 increments), covering
its full environment action range for the selected mission. Rerun this training
cell to learn with the wider command range; existing checkpoints retain their
original saved `commands.npy`. The Results cell reports peak altitude so you can
check whether the trained policy reaches its target.

Use learning rate `1e-4`, batch `128`, `train_freq=5`, and `gradient_steps=-1`:
with 10 environments, 50 collected transitions are followed by 50 updates.
`n_steps=3` passes rewards across three transitions; `tau=0.1` and
`target_update_interval=1000` smooth the target updates.
`exploration_fraction=1.0` decays epsilon throughout the training run.
These choices target sparse landing rewards and the observed training regressions.

Sources: [DQN parameters](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters),
[multi-environment update guidance](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html#multiprocessing-with-off-policy-algorithms).


In [ ]:
if SKIP_DQN:
    print("DQN skipped (SKIP_DQN=True).")
else:
    def make_dqn_env():
        env = DiscreteDroneActions(DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission))
        return env

    dqn_env = make_vec_env(make_dqn_env, n_envs=10, seed=SEED, monitor_dir=str(DQN_DIR))
    dqn_eval = make_vec_env(make_dqn_env, n_envs=1, seed=100)
    dqn_callback = EvalCallback(
        dqn_eval, best_model_save_path=str(DQN_DIR), log_path=str(DQN_DIR),
        eval_freq=500, n_eval_episodes=1, deterministic=True, verbose=0,
        callback_on_new_best=StopTrainingOnRewardThreshold(reward_threshold=0.0, verbose=1),
    )
    dqn_agent = DQN("MlpPolicy", dqn_env, seed=SEED, device="cpu", verbose=0, **DQN_PARAMS)
    start = time.perf_counter()
    dqn_agent.learn(total_timesteps=DQN_STEPS, callback=dqn_callback)
    dqn_agent.save(DQN_DIR / "dqn_drone_rnn")
    np.save(DQN_DIR / "commands.npy", DQN_COMMANDS)
    DQN_MODEL_PATH = DQN_DIR / "best_model.zip"
    dqn_env.close()
    dqn_eval.close()
    print(f"DQN: {dqn_agent.num_timesteps:,} steps in {time.perf_counter() - start:.1f} s; {DQN_MODEL_PATH}")


## DQN — Results

Skipped when `SKIP_DQN=True` in Setup.

Loads only DQN and its saved `commands.npy`; PPO and SAC do not need to run.


In [ ]:
if SKIP_DQN:
    print("DQN skipped (SKIP_DQN=True).")
else:
    show_result("DQN", DQN_MODEL_PATH)


## SAC — optional training

Skipped when `SKIP_SAC=True` in Setup.

Use learning rate `3e-4`, batch `128`, and `gradient_steps=-1` to match updates
to collected transitions. Explore with `use_sde=True`, `sde_sample_freq=4`,
and `log_std_init=-2`: noise is resampled every 0.2 s instead of independently
at each 0.05 s action. Start entropy tuning at `auto_0.05` to reduce its initial
weight relative to the reward. The notebook wrapper advertises commands in
`[-0.8, 0.8]`; physical command units and library dynamics stay unchanged.

The best 30k-step checkpoint landed at 7.20 s. Later training regressed, so the
Results cell loads that best checkpoint, not the final 50k-step policy.
These values were tested together; their individual effects are unproven.
Sources: [SAC parameters](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html#parameters),
[Smooth Exploration for Robotic Reinforcement Learning](https://arxiv.org/abs/2005.05719),
[RL Zoo SAC configurations](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/sac.yml).
The sources support the approach; they do not prescribe these exact values for this drone.


In [ ]:
if SKIP_SAC:
    print("SAC skipped (SKIP_SAC=True).")
else:
    def make_sac_env():
        env = DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission)
        if SAC_ACTION_LIMIT is not None:
            env = LimitedCommands(env, SAC_ACTION_LIMIT)
        return env

    sac_env = make_vec_env(make_sac_env, n_envs=10, seed=SEED, monitor_dir=str(SAC_DIR))
    sac_eval = make_vec_env(make_sac_env, n_envs=1, seed=100)
    sac_callback = EvalCallback(
        sac_eval, best_model_save_path=str(SAC_DIR), log_path=str(SAC_DIR),
        eval_freq=500, n_eval_episodes=1, deterministic=True, verbose=0,
        callback_on_new_best=StopTrainingOnRewardThreshold(reward_threshold=0.0, verbose=1),
    )
    sac_agent = SAC("MlpPolicy", sac_env, seed=SEED, device="cpu", verbose=0, **SAC_PARAMS)
    start = time.perf_counter()
    sac_agent.learn(total_timesteps=SAC_STEPS, callback=sac_callback)
    sac_agent.save(SAC_DIR / "sac_drone_rnn")
    SAC_MODEL_PATH = SAC_DIR / "best_model.zip"
    sac_env.close()
    sac_eval.close()
    print(f"SAC: {sac_agent.num_timesteps:,} steps in {time.perf_counter() - start:.1f} s; {SAC_MODEL_PATH}")


## SAC — Results

Skipped when `SKIP_SAC=True` in Setup.

Loads only SAC; PPO and DQN do not need to run.


In [ ]:
if SKIP_SAC:
    print("SAC skipped (SKIP_SAC=True).")
else:
    show_result("SAC", SAC_MODEL_PATH)


## Export a PPO trajectory

Exports `trajectory_az.csv` into this run's PPO directory using the same selected
mission. MATLAB policy export above also includes that mission's reference samples.


In [ ]:

# 1. Déroulement d'un épisode avec la politique entraînée
eval_env = DroneEnv(rnn_model=acceleration_rnn, mission=ref_mission)
obs, info = eval_env.reset(seed=0)

az_list = [float(eval_env.unwrapped.state[2])]

terminated = False
truncated = False

while not (terminated or truncated):
    action, _ = agent.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    az_list.append(float(eval_env.unwrapped.state[2]))

eval_env.close()

# 2. Export sur une seule ligne (chaque case = une valeur de az)
LOG_DIR.mkdir(parents=True, exist_ok=True)
output_path = LOG_DIR / "trajectory_az.csv"
np.savetxt(output_path, [az_list], delimiter=",", fmt="%.6f")

print(f"Trajectoire az exportée ({len(az_list)} valeurs sur 1 ligne) : {output_path}")